In [1]:

! pip install pandas regex pyspellchecker

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [2]:
! pip install indic-nlp-library

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [17]:
!pip install wordfreq
!pip install pyspellchecker

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable
Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [18]:
import pandas as pd
import re
from wordfreq import zipf_frequency

In [19]:
INPUT_PATH  = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs\Unique Words Data - Sheet1.csv"
OUTPUT_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q3_Spelling_Correction\final_spelling_output.csv"

In [20]:
df = pd.read_csv(INPUT_PATH)

# Rename first column to 'word' regardless of original name
df.columns = ['word'] + list(df.columns[1:])

print("Total words loaded:", len(df))
df.head()

Total words loaded: 177508


,word
0,है
1,तो
2,में
3,जी
4,हैं


In [21]:
def clean_word(word):
    word = str(word).strip()
    # Keep only Devanagari characters
    word = re.sub(r'[^\u0900-\u097F]', '', word)
    return word

df["clean_word"] = df["word"].apply(clean_word)

# Remove empty and duplicates
df = df[df["clean_word"] != ""]
df = df.drop_duplicates(subset=["clean_word"])
df.reset_index(drop=True, inplace=True)

print("After cleaning:", len(df))

After cleaning: 162211


In [ ]:
# from wordfreq import zipf_frequency
# import re

# # Only these are genuinely incorrect
# NOISE_PATTERN = re.compile(
#     r'^[हमशआउऊइई्]{1,4}([म्हश]{1,2}){2,}$'
# )

# # Compound words without space (अलगअलग, कभीकभी)
# # These contain a full Hindi word repeated or joined
# KNOWN_CORRECT_WORDS = set([
#     'अलग', 'कभी', 'धीरे', 'साथ', 'कहीं', 'यहां', 'वहां'
# ])

# def is_compound_no_space(word):
#     # If word is 8+ chars and same sub-pattern repeats — likely missing space
#     if len(word) >= 8:
#         half = len(word) // 2
#         first_half = word[:half]
#         second_half = word[half:]
#         if first_half == second_half:
#             return True
#         if first_half in KNOWN_CORRECT_WORDS and second_half in KNOWN_CORRECT_WORDS:
#             return True
#     return False

# def classify_word(word):

#     # Rule 1: Known frequency — correct
#     freq = zipf_frequency(word, "hi")
#     if freq >= 1.0:
#         return "correct spelling"

#     # Rule 2: Short words — correct
#     if len(word) <= 3:
#         return "correct spelling"

#     # Rule 3: Filler/noise sounds — incorrect (ह्म्म्म, हम्मम)
#     if NOISE_PATTERN.match(word):
#         return "incorrect spelling"

#     # Rule 4: All same chars repeated — incorrect (हाहाहा)
#     unique_chars = set(list(word))
#     if len(unique_chars) <= 2 and len(word) >= 5:
#         return "incorrect spelling"

#     # Rule 5: Compound words joined without space — incorrect
#     if is_compound_no_space(word):
#         return "incorrect spelling"

#     # Rule 6: Everything else — CORRECT
#     # This covers:
#     # English in Devanagari (क्वेश्चन, मोटिवेट, थैंक्यू) ✅
#     # Rare Hindi words ✅
#     # Names, places ✅
#     return "correct spelling"

# df["spelling_status"] = df["clean_word"].apply(classify_word)

In [42]:
from wordfreq import zipf_frequency
import re

NOISE_PATTERN = re.compile(r'^[हमशआउऊइई्]{1,4}([म्हश]{1,2}){2,}$')

VALID_HINDI_SUFFIXES = [
    'ना', 'ता', 'ती', 'ते', 'या', 'यी', 'ये', 'कर',
    'गा', 'गी', 'गे', 'ने', 'में', 'को', 'से', 'पर',
    'है', 'हैं', 'था', 'थी', 'थे', 'ओं', 'िया', 'ियां',
    'वाला', 'वाली', 'वाले', 'पन', 'कार', 'दार', 'बाज'
]

VALID_ENGLISH_DEVA_SUFFIXES = [
    'िंग', 'शन', 'टेड', 'र्स', 'ली', 'री', 'नी',
    'यू', 'क्यू', 'थ', 'ट्स', 'ड्स', 'िस्ट', 'इज्म'
]

ENGLISH_DEVA_PREFIX = re.compile(
    r'^(एक्स|इन्|कं|रि|डि|स्ट|ट्र|नॉ|मो|थै|ट्व|'
    r'क्व|पो|फो|शो|मि|वि|सि|ले|रे|बे|मोटि|'
    r'नॉर्|ट्वे|थैं|क्वे|ट्यू|ड्यू|ब्यू|'
    r'प्रो|स्पे|ग्र|ब्र|ड्र|क्र|फ्र|'
    r'अन्|उन्|इम्|कम्|सम्|'
    r'ब्ल|ब्लै|इं|इंपो|इंस्|इंट|'  # ← ADD THIS LINE
    r'फ्ल|स्ल|स्क|स्प|स्म|स्न)'
)

def has_valid_suffix(word):
    all_suffixes = VALID_HINDI_SUFFIXES + VALID_ENGLISH_DEVA_SUFFIXES
    for suffix in all_suffixes:
        if word.endswith(suffix):
            return True
    return False

def is_compound_no_space(word):
    if len(word) >= 8:
        half = len(word) // 2
        if word[:half] == word[half:]:
            return True
        # Check if two valid Hindi words joined
        for split in range(3, len(word)-2):
            part1 = word[:split]
            part2 = word[split:]
            freq1 = zipf_frequency(part1, "hi")
            freq2 = zipf_frequency(part2, "hi")
            if freq1 >= 2.0 and freq2 >= 2.0:
                return True
    return False

def classify_word(word):

    freq = zipf_frequency(word, "hi")

    # Rule 1: Known Hindi words — correct
    if freq >= 1.5:
        return "correct spelling"

    # Rule 2: Short words — correct
    if len(word) <= 2:
        return "correct spelling"

    # Rule 3: Filler/noise — incorrect
    if NOISE_PATTERN.match(word):
        return "incorrect spelling"

    # Rule 4: Repeated chars — incorrect
    if len(set(list(word))) <= 2 and len(word) >= 4:
        return "incorrect spelling"

    # Rule 5: Compound words without space — incorrect
    if is_compound_no_space(word):
        return "incorrect spelling"

    # Rule 6: English in Devanagari prefix — correct
    if ENGLISH_DEVA_PREFIX.match(word):
        return "correct spelling"

    # Rule 7: Valid suffix — correct
    if has_valid_suffix(word):
        return "correct spelling"

    # Rule 8: Low-medium frequency — correct
    if freq >= 0.5:
        return "correct spelling"

    # Rule 9: Zero frequency + long word = likely misspelled
    if freq == 0 and len(word) >= 7:
        return "incorrect spelling"

    # Rule 10: Short-medium unknown words — correct
    if len(word) <= 5:
        return "correct spelling"

    return "incorrect spelling"

df["spelling_status"] = df["clean_word"].apply(classify_word)

In [43]:
total      = len(df)
correct    = (df["spelling_status"] == "correct spelling").sum()
incorrect  = (df["spelling_status"] == "incorrect spelling").sum()

print("=" * 40)
print("Total unique words   :", total)
print("Correct spelling     :", correct)
print("Incorrect spelling   :", incorrect)
print("=" * 40)

Total unique words   : 162211
Correct spelling     : 114082
Incorrect spelling   : 48129


In [44]:
import os

# Create folder if not exists
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

final_df = df[["clean_word", "spelling_status"]]
final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("File saved successfully!")
print("Location:", OUTPUT_PATH)
print("Total rows saved:", len(final_df))

File saved successfully!
Location: C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q3_Spelling_Correction\final_spelling_output.csv
Total rows saved: 162211


In [45]:
print("\nSample CORRECT words:")
print(df[df["spelling_status"] == "correct spelling"]["clean_word"].head(10).tolist())

print("\nSample INCORRECT words:")
print(df[df["spelling_status"] == "incorrect spelling"]["clean_word"].head(10).tolist())


Sample CORRECT words:
['है', 'तो', 'में', 'जी', 'हैं', 'भी', 'के', 'नहीं', 'कि', 'वो']

Sample INCORRECT words:
['ह्म्म', 'उम्म', 'अलगअलग', 'कभीकभी', 'हम्मम', 'जीजी', 'ह्म्म्म', 'हम्म्म', 'एक्टिविटी', 'पॉसिबल']


We performed script normalization to retain only valid Devanagari tokens and removed noise characters. Duplicate words were eliminated to ensure unique entries. We applied a frequency-based validation strategy using Hindi corpus statistics (Zipf frequency). Words above a balanced frequency threshold were classified as correct spelling, while rare or irregular tokens were classified as incorrect spelling. This hybrid rule + frequency-based approach ensured scalability while maintaining linguistic validity.